# Level 1 - Classic CNNs

Source notebook: `level1_classic_cnns_2026181713_최민경.ipynb`


# Level 1 — Classic CNNs (VGG-16, ResNet-18 / 50)

**목표**: VGG 와 ResNet 을 직접 구현하여 Set A 위에서 Multi-task 로 학습하고, **Skip Connection** 이 깊은 네트워크의 수렴에 미치는 영향을 분석합니다.

**금지 사항**: `torchvision.models.*`, `timm`. 단, 위 라이브러리의 코드를 *참고하여 직접 타이핑* 하는 것은 허용됩니다.

본 노트북의 산출물:
1. 학습된 체크포인트 (`checkpoints/level1_*.pth`).
2. VGG-16, ResNet-18, ResNet-50 의 `Avg-Macro-F1` 및 속성별 Macro-F1 표.
3. VGG (skip 없음) vs ResNet (skip 있음) 의 학습 손실 곡선 비교.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Reproducibility entry point.
# Code is loaded from the student's GitHub repository.
# Large checkpoint files are downloaded later from the shared Google Drive folder.
REPO_URL = "https://github.com/cmg3931-tech/2026-HYU-AUE8088-PA2-2026181713.git"
REPO_NAME = "2026-HYU-AUE8088-PA2-2026181713"

if Path("/content").exists():
    # Colab: clone or update the final student repository under /content.
    REPO_ROOT = Path("/content") / REPO_NAME
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull"], check=True)
else:
    # Local or VS Code: infer the repository root from the current directory.
    REPO_ROOT = Path.cwd().resolve()
    if REPO_ROOT.name == "notebooks":
        REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Repo URL:", REPO_URL)


Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 36 (delta 3), reused 0 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (36/36), 47.09 KiB | 4.71 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-HYU-AUE8088-PA2


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16

SEED = 42
set_seed(SEED, deterministic=True)   # 재현성을 위한 시드 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Weights & Biases 설정

학습 곡선과 속성별 Macro-F1 을 자동으로 로깅합니다. 사용하지 않으려면 `WANDB_PROJECT = None` 으로 두세요 — 로깅 호출은 자동으로 no-op 이 됩니다.

최초 1회만:
```python
import wandb; wandb.login()   # API key 입력
```

In [ ]:
!pip install -q --upgrade wandb

In [ ]:
import wandb; wandb.login()   # API key 입력

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmg125 (cmg_125) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level1"]
# 환경변수로도 끌 수 있음:  os.environ["WANDB_DISABLED"] = "true"

In [ ]:
DATA_ROOT = "data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# data/set_a 가 없으면 zip 을 받아 현재 repo root에 압축 해제 → data/set_a, data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "aue8088_pa2_data.zip"
EXTRACT_TO = "."   # zip 내부 최상위가 data/ 이므로 repo root에 풀면 data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# Set A 의 train/val 로더 구성
train_ds = BDDAttrDataset(DATA_ROOT, split="train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, split="val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# 속성별 클래스 분포 출력 — 불균형 정도를 직접 확인할 것
for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=cd824478-c480-430b-a17a-7a5236fa1c1c
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:04<00:00, 49.8MB/s]


압축 해제 중...
완료 → ../data/set_a
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [ ]:
from torch import nn

def train_one(model_fn, name, epochs=20):
    """단일 모델을 학습하고 체크포인트와 wandb 산출물을 저장한다."""
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
    cfg = TrainConfig(epochs=epochs)

    # wandb 로거 — WANDB_PROJECT 가 None 이거나 wandb 미설치 시 자동 no-op
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name, "epochs": epochs, "batch": BATCH,
            "lr": 3e-4, "weight_decay": 5e-4, "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    # 학습 종료 후 — 속성별 정규화 confusion matrix 를 wandb 에 업로드
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    logger.finish()

    os.makedirs("checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history},
               f"checkpoints/level1_{name}.pth")
    return model, history

# TODO: 각 모델을 학습하고 최종 Avg-Macro-F1 및 속성별 MF1 을 기록하세요.
#vgg_model, vgg_hist = train_one(VGG16,    "vgg16",    epochs=30)
#r18_model, r18_hist = train_one(resnet18, "resnet18", epochs=30)
r50_model, r50_hist = train_one(resnet50, "resnet50", epochs=30)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.4377  val_avg_MF1=0.4182  per={'weather': 0.228331253360931, 'scene': 0.3945657094277544, 'timeofday': 0.6318252004145779}


[epoch 02/30] train_loss=2.0562  val_avg_MF1=0.4175  per={'weather': 0.13169239326936447, 'scene': 0.37127672877328993, 'timeofday': 0.7493915016410124}


[epoch 03/30] train_loss=1.9760  val_avg_MF1=0.4835  per={'weather': 0.21555815517970148, 'scene': 0.43924885806124175, 'timeofday': 0.7955890342569344}


[epoch 04/30] train_loss=1.8772  val_avg_MF1=0.3383  per={'weather': 0.2915029668857468, 'scene': 0.3521583167540845, 'timeofday': 0.3713788806542124}


...

[epoch 27/30] train_loss=1.1270  val_avg_MF1=0.6373  per={'weather': 0.5180809053854146, 'scene': 0.5632098190831808, 'timeofday': 0.8305974555568499}


[epoch 28/30] train_loss=1.1122  val_avg_MF1=0.6289  per={'weather': 0.5016517957100509, 'scene': 0.5770252042621059, 'timeofday': 0.8080850707226701}


[epoch 29/30] train_loss=1.0778  val_avg_MF1=0.6309  per={'weather': 0.5113012123206998, 'scene': 0.5734574603297703, 'timeofday': 0.8080850707226701}


[epoch 30/30] train_loss=1.0779  val_avg_MF1=0.6250  per={'weather': 0.5105002199249961, 'scene': 0.5543904976437094, 'timeofday': 0.8100848542393301}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val/avg_macro_f1,▃▃▄▁▄▄▁▅▅▆▅▆▇▆▅▆▇▆▇▆██▇█▇▇████
val/mf1_scene,▄▃▅▃▂▄▁▄▄▅▅▄▆▅▄▄▅▃▅▅██▆██▇███▇
val/mf1_timeofday,▅▇▇▁▇▇▂██▇▇██▇▇▇███▇██▇███████
val/mf1_weather,▃▁▃▄▄▂▃▄▄▆▄▅▇▅▄▇▇▇▇▇█▇██▇█████
epoch,30
lr,0
train/loss,1.07788
val/avg_macro_f1,0.62499


In [ ]:
from torch import nn

def train_one_weighted(model_fn, name, epochs=20, loss_weights=None):
    set_seed(SEED, deterministic=True)
    model = model_fn().to(device)

    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

    if loss_weights is None:
        loss_weights = {
            "weather": 1.0,
            "scene": 1.0,
            "timeofday": 1.0,
        }

    cfg = TrainConfig(
        epochs=epochs,
        loss_weights=loss_weights,
    )

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level1-{name}",
        config={
            "backbone": name,
            "epochs": epochs,
            "batch": BATCH,
            "lr": 3e-4,
            "weight_decay": 5e-4,
            "seed": SEED,
            "loss_weights": cfg.loss_weights,
        },
        tags=WANDB_TAGS + [name],
    )

    trainer = MultiTaskTrainer(model, optim, sched, losses, device, cfg, logger=logger)
    history = trainer.fit(train_loader, val_loader)

    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)

    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])

    logger.finish()

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "history": history,
            "loss_weights": cfg.loss_weights,
        },
        f"checkpoints/level1_{name}.pth",
    )

    return model, history


r18_w211_model, r18_w211_hist = train_one_weighted(
    resnet18,
    "resnet18_w211",
    epochs=30,
    loss_weights={
        "weather": 2.0,
        "scene": 1.0,
        "timeofday": 1.0,
    },
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=3.0594  val_avg_MF1=0.4594  per={'weather': 0.3246589830214816, 'scene': 0.38288293583157995, 'timeofday': 0.6705098844402326}


[epoch 02/30] train_loss=2.7120  val_avg_MF1=0.4588  per={'weather': 0.2220734764307749, 'scene': 0.4222266903477319, 'timeofday': 0.7320258621515149}


[epoch 03/30] train_loss=2.6525  val_avg_MF1=0.4643  per={'weather': 0.25832269599371616, 'scene': 0.3655415419048483, 'timeofday': 0.7690339776327036}


[epoch 04/30] train_loss=2.5404  val_avg_MF1=0.4839  per={'weather': 0.2518202156163029, 'scene': 0.48038760017464543, 'timeofday': 0.7194267878117149}


...

[epoch 27/30] train_loss=1.5556  val_avg_MF1=0.6337  per={'weather': 0.5190980221648086, 'scene': 0.5835978835978837, 'timeofday': 0.7983877362230739}


[epoch 28/30] train_loss=1.5434  val_avg_MF1=0.6319  per={'weather': 0.5208253062125662, 'scene': 0.5879754168749028, 'timeofday': 0.7868262411347517}


[epoch 29/30] train_loss=1.4984  val_avg_MF1=0.6373  per={'weather': 0.5374096669865457, 'scene': 0.5760833907169548, 'timeofday': 0.7983877362230739}


[epoch 30/30] train_loss=1.5050  val_avg_MF1=0.6387  per={'weather': 0.5331508981684363, 'scene': 0.585179552459213, 'timeofday': 0.7976747204011975}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val/avg_macro_f1,▂▂▂▃▄▁▅▅▄▅▅▅▅▅▃▇▆▄▇▆▇▇▇█▇█████
val/mf1_scene,▂▃▂▅▃▃▄▃▄▄▂▄▂▄▁▅▄▁▆▅▇▇▆█▆▇████
val/mf1_timeofday,▁▄▆▃▅▃▇▇▅▇▇▄▇▇▄█▇▆▇▆█▇▇█▇▇▇▆▇▇
val/mf1_weather,▄▂▃▂▄▁▆▆▃▅▆▇▇▆▅█▇▆▇▆▇▆█▇▇█████
epoch,30
lr,0
train/loss,1.50498
val/avg_macro_f1,0.63867


## 분석 (리포트 필수 포함 항목)

1. **수렴 비교**: VGG-16 과 ResNet-18 의 학습 손실 곡선을 함께 그리세요. Skip connection 이 없는 깊은 네트워크가 정체되는 현상이 관찰되는지, 그 원인은 무엇인지 서술하세요.
2. **속성별 MF1 표**: 각 백본에 대해 Weather / Scene / Time of Day 의 Macro-F1 을 표로 정리하세요. 어느 속성이 가장 어려운지, 깊이 변화에 따라 그 양상이 어떻게 바뀌는지 분석하세요.
3. **Loss 가중치 민감도**: ResNet-18 에 대해 최소 두 가지 비자명한 가중치 설정 (예: `(1, 1, 1)` vs `(2, 1, 1)`) 으로 재학습하고 Avg-MF1 변화를 보고하세요.

wandb 를 활성화한 경우 같은 프로젝트의 Run 들을 비교하면 학습 곡선·속성별 MF1·confusion matrix 가 한눈에 정리됩니다.

# Level 2 - Transformers

Source notebook: `level2_transformers_2026181713_최민경.ipynb`


# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [ ]:
import os
import sys
from pathlib import Path

# 제출 폴더 기준으로 repo root 설정
REPO_ROOT = Path.cwd().resolve()

# 노트북을 notebooks/ 안에서 실행한 경우 한 단계 위를 root로 사용
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
assert os.path.exists("requirements.txt"), "requirements.txt not found. REPO_ROOT 설정을 확인하세요."
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 36 (delta 3), reused 0 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (36/36), 47.09 KiB | 6.73 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmg125 (cmg_125) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
DATA_ROOT = "data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# data/set_a 가 없으면 zip 을 받아 현재 repo root에 압축 해제 → data/set_a, data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "aue8088_pa2_data.zip"
EXTRACT_TO = "."   # zip 내부 최상위가 data/ 이므로 repo root에 풀면 data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=f11100a2-a977-44a9-9fe7-3a23b152476a
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 118MB/s]


압축 해제 중...
완료 → ../data/set_a


In [7]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
USE_PRETRAINED = False
model = vit_small_patch16_224().to(device)

In [ ]:
epochs = 25
optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level2-vit_s16{'-pretrained' if USE_PRETRAINED else ''}",
    config={
        "backbone": "vit_s16", "pretrained": USE_PRETRAINED,
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16"],
)
trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)

history = trainer.fit(train_loader, val_loader)

val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

os.makedirs("checkpoints", exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "pretrained": USE_PRETRAINED,
    },
    "checkpoints/level2_vit_s16.pth",
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/25] train_loss=2.3619  val_avg_MF1=0.3884  per={'weather': 0.20039308679954712, 'scene': 0.3264927196325613, 'timeofday': 0.6383568122698557}


[epoch 02/25] train_loss=2.0853  val_avg_MF1=0.3829  per={'weather': 0.13790737564322472, 'scene': 0.372599591925675, 'timeofday': 0.6382483583987933}


[epoch 03/25] train_loss=2.0448  val_avg_MF1=0.3956  per={'weather': 0.220358677540998, 'scene': 0.32804897649070003, 'timeofday': 0.6383052078432038}


[epoch 04/25] train_loss=1.9969  val_avg_MF1=0.4061  per={'weather': 0.2358392622766546, 'scene': 0.3454654157468727, 'timeofday': 0.6371309443317558}


...

[epoch 22/25] train_loss=1.7033  val_avg_MF1=0.5113  per={'weather': 0.36517341012829124, 'scene': 0.3867258879863417, 'timeofday': 0.7819035196392315}


[epoch 23/25] train_loss=1.6847  val_avg_MF1=0.5297  per={'weather': 0.3886836151095676, 'scene': 0.4034860947904426, 'timeofday': 0.7970666630168539}


[epoch 24/25] train_loss=1.6787  val_avg_MF1=0.5278  per={'weather': 0.37378753431164585, 'scene': 0.405395114020451, 'timeofday': 0.8042242433632628}


[epoch 25/25] train_loss=1.6623  val_avg_MF1=0.5268  per={'weather': 0.3679273326232501, 'scene': 0.4081820624016579, 'timeofday': 0.8042242433632628}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val/avg_macro_f1,▁▁▂▂▂▃▅▃▄▄▅▄▆▅▆▆▆▆▇▇▆▇███
val/mf1_scene,▄▆▄▅▅▆▇▁▆▅▆▅▆▄▇▇▇▇▇▆▆▇███
val/mf1_timeofday,▁▁▁▁▁▄▅▅▅▃▅▅▇▆▇▆▇█▇█▇▇███
val/mf1_weather,▃▁▃▄▄▃▅▅▄▆▅▅▆▇▆▆▆▆▇▇▇▇██▇
epoch,25
lr,0
train/loss,1.66232
val/avg_macro_f1,0.52678


## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.

# Level 3 - Imbalance Handling

Source notebook: `level3_imbalance_2026181713_최민경.ipynb`


# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys
from pathlib import Path

# 제출 폴더 기준으로 repo root 설정
REPO_ROOT = Path.cwd().resolve()

# 노트북을 notebooks/ 안에서 실행한 경우 한 단계 위를 root로 사용
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

#%load_ext autoreload
#%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
assert os.path.exists("requirements.txt"), "requirements.txt not found. REPO_ROOT 설정을 확인하세요."
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 36 (delta 3), reused 0 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (36/36), 47.09 KiB | 790.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 M

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.resnet import resnet18

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]
# 각 실험마다 RUN_NAME 만 바꿔서 동일 프로젝트에 누적하세요.
EXPERIMENT_NAME = "resnet18_focal_weather_sampler"

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmg125 (cmg_125) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
DATA_ROOT = "data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# data/set_a 가 없으면 zip 을 받아 현재 repo root에 압축 해제 → data/set_a, data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "aue8088_pa2_data.zip"
EXTRACT_TO = "."   # zip 내부 최상위가 data/ 이므로 repo root에 풀면 data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 옵션 A — 가장 불균형이 심한 weather 속성 기준 class-balanced sampler 사용
sampler = class_balanced_sampler(train_ds, attribute="weather")
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=db9a55c4-9612-41e1-be5f-0d61593a899e
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 96.2MB/s]


압축 해제 중...
완료 → ../data/set_a


In [ ]:
from importlib import reload
import src.models.resnet

reload(src.models.resnet)

from src.models.resnet import resnet18, resnet50

# 옵션 B — 속성별로 다른 loss 적용. 가장 불균형이 심한 속성에 가장 강한 loss 사용.
samples_s = train_ds.class_counts("scene")

loss_fns = {
    "weather":   FocalLoss(gamma=2.0).to(device),
    "scene":     ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}

model = resnet18().to(device)
epochs = 25
optim  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "backbone": "resnet18",
        "sampler": "class_balanced(weather)",
        "loss": {"weather": "focal_g2.0", "scene": "cb_loss", "timeofday": "ce"},
        "epochs": epochs, "batch": BATCH, "lr": 3e-4, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME],
)
trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [10]:
# 옵션 C — 학습 루프에 Mixup/CutMix 를 통합하여 적용
# (깨끗한 실험을 위해서는 _train_one_epoch 를 서브클래싱하는 것이 좋으나,
#  아래는 augmented step 의 핵심만 인라인으로 보인 것입니다.)

from tqdm import tqdm

def step_with_mix(images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)

    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

history = {
    "train_loss": [],
    "val_avg_mf1": [],
    "val_per_mf1": [],
}

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"train e{epoch + 1}", leave=False)

    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        targets = {
            a: batch[a].to(device, non_blocking=True)
            for a in ATTRIBUTES
        }

        optim.zero_grad(set_to_none=True)

        loss = step_with_mix(images, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

        running_loss += loss.item()
        n_batches += 1

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss = running_loss / max(n_batches, 1)
    val_metrics = trainer.evaluate(val_loader)

    history["train_loss"].append(train_loss)
    history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
    history["val_per_mf1"].append(val_metrics["per_macro_f1"])

    sched.step()

    print(
        f"[epoch {epoch + 1:02d}/{epochs}] "
        f"train_loss={train_loss:.4f} "
        f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f} "
        f"per={val_metrics['per_macro_f1']}"
    )

    log_payload = {
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "val/avg_macro_f1": val_metrics["avg_macro_f1"],
        "lr": optim.param_groups[0]["lr"],
    }

    for a, v in val_metrics["per_macro_f1"].items():
        log_payload[f"val/mf1_{a}"] = v

    logger.log(log_payload, step=epoch)

[epoch 01/25] train_loss=2.4902 val_avg_MF1=0.5035 per={'weather': 0.3266925618684988, 'scene': 0.437575139906676, 'timeofday': 0.7462644901051182}


[epoch 02/25] train_loss=2.3948 val_avg_MF1=0.4538 per={'weather': 0.28122783932953427, 'scene': 0.44136530398322843, 'timeofday': 0.6388973465498382}


[epoch 03/25] train_loss=2.3007 val_avg_MF1=0.5334 per={'weather': 0.3811881547256319, 'scene': 0.4686194021427618, 'timeofday': 0.7504524535911888}


[epoch 04/25] train_loss=2.2835 val_avg_MF1=0.4624 per={'weather': 0.29827475157272865, 'scene': 0.44939934708834955, 'timeofday': 0.639636101009845}


[epoch 05/25] train_loss=2.1775 val_avg_MF1=0.5432 per={'weather': 0.4381018465875071, 'scene': 0.44076921788364576, 'timeofday': 0.7507957185137268}


...

[epoch 21/25] train_loss=1.6716 val_avg_MF1=0.6459 per={'weather': 0.5413781020271893, 'scene': 0.6025375559107204, 'timeofday': 0.7938778451937929}


[epoch 22/25] train_loss=1.7601 val_avg_MF1=0.6405 per={'weather': 0.5219576766603465, 'scene': 0.6166727812732496, 'timeofday': 0.7828230768101259}


[epoch 23/25] train_loss=1.7463 val_avg_MF1=0.6309 per={'weather': 0.5327457911120094, 'scene': 0.5995446124010874, 'timeofday': 0.7602644774775923}


[epoch 24/25] train_loss=1.7286 val_avg_MF1=0.6312 per={'weather': 0.5222109727460745, 'scene': 0.5908942864582012, 'timeofday': 0.780518034587956}


[epoch 25/25] train_loss=1.6354 val_avg_MF1=0.6453 per={'weather': 0.5349151374380732, 'scene': 0.6105508127613176, 'timeofday': 0.7905557516499765}


In [ ]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])

    rows = list(zip(
        prf[a]["class"],
        prf[a]["precision"],
        prf[a]["recall"],
        prf[a]["f1"],
        prf[a]["support"],
    ))

    logger.log_table(
        f"final/prf_{a}",
        ["class", "P", "R", "F1", "support"],
        [list(r) for r in rows],
    )

os.makedirs("checkpoints", exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "experiment": EXPERIMENT_NAME,
        "sampler": "class_balanced(weather)",
        "loss": {
            "weather": "FocalLoss(gamma=2.0)",
            "scene": "ClassBalancedLoss",
            "timeofday": "CrossEntropyLoss",
        },
    },
    "checkpoints/level3_best.pth",
)

logger.finish()

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▆▆▅▆▅▄▅▅▄▄▄▃▄▃▃▁▂▂▁▂▂▂▁
val/avg_macro_f1,▃▁▄▁▄▃▄▂▄▆▅▅▅▇▄▇▇▆█▇██▇▇█
val/mf1_scene,▁▁▂▂▁▂▂▁▃▅▄▆▃▇▂▇▆▇█▆▇█▇▇█
val/mf1_timeofday,▆▁▆▁▆▃█▄▃█▇▃█▇▆▆▆▅████▆▇█
val/mf1_weather,▃▁▄▂▅▃▃▁▅▆▆▅▄▆▅▆▇▆▇▇█████
epoch,25
lr,0
train/loss,1.6354
val/avg_macro_f1,0.64534


## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.

# Level 4 - XAI & Efficiency

Source notebook: `level4_xai_efficiency.ipynb`


In [ ]:
# Download checkpoints from the shared Google Drive folder if they are not present.
# Level 1 and Level 2 are trained from scratch in this notebook.
# These checkpoints are only used for Level 4 analysis and Level 5 fine-tuning/evaluation.
from pathlib import Path
import subprocess
import sys

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Google Drive folder containing .pth files. The folder must be shared as: Anyone with the link -> Viewer.
DRIVE_CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/12M5hTVH6cWgfRwXqMc7LlBB-qwNvTYmJ?usp=drive_link"

REQUIRED_CHECKPOINTS = [
    "level1_resnet18.pth",   # Level 4 best-base analysis + Level 5 scoring/fine-tuning init
    "level5_final.pth",      # Final Level 5 model checkpoint for reproducible evaluation
]

missing = [name for name in REQUIRED_CHECKPOINTS if not (CHECKPOINT_DIR / name).exists()]

if missing:
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    print("Missing checkpoints:", missing)
    print("Downloading checkpoints from Drive folder...")
    gdown.download_folder(
        url=DRIVE_CKPT_FOLDER_URL,
        output=str(CHECKPOINT_DIR),
        quiet=False,
        use_cookies=False,
    )

still_missing = [name for name in REQUIRED_CHECKPOINTS if not (CHECKPOINT_DIR / name).exists()]
assert not still_missing, (
    "checkpoint download failed or Drive sharing is not public: "
    f"{still_missing}. Check that the Drive folder and files are shared as Anyone with the link -> Viewer."
)

print("Checkpoint dir:", CHECKPOINT_DIR.resolve())
print("Available checkpoints:", sorted([p.name for p in CHECKPOINT_DIR.glob("*.pth")]))


# Level 4 — 심층 분석: Grad-CAM 및 효율성 Trade-off

**목표**: 모델의 *동작 원리* 를 설명하고, FPS와 정확도의 trade-off 를 정량화합니다.

리포트 필수 산출물:
1. **속성별 정규화 Confusion Matrix 3개** — best 모델 기준.
2. **Grad-CAM 패널** — 같은 이미지에 대해 3개 head 가 각각 어디를 보는지 시각화 (예: `rainy + night + city street` 인 이미지).
3. 모든 백본에 대한 **FPS vs Avg-MF1 Pareto plot**.

본 노트북은 학습 노트북이 아니라 분석 노트북이지만, wandb 가 활성화되어 있으면 confusion matrix 이미지·Grad-CAM 패널·FPS 표를 같은 프로젝트의 별도 Run 으로 업로드합니다.

In [ ]:
import os
import sys
from pathlib import Path

# 제출 폴더 기준으로 repo root 설정
REPO_ROOT = Path.cwd().resolve()

# 노트북을 notebooks/ 안에서 실행한 경우 한 단계 위를 root로 사용
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
assert os.path.exists("requirements.txt"), "requirements.txt not found. REPO_ROOT 설정을 확인하세요."
!pip install -q -r requirements.txt

# Level 4 분석에 필요한 checkpoint 확인
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
BEST_CKPT = CHECKPOINT_DIR / "level1_resnet18.pth"

assert CHECKPOINT_DIR.exists(), f"checkpoint directory not found: {CHECKPOINT_DIR}"
assert BEST_CKPT.exists(), f"checkpoint not found: {BEST_CKPT}"

print("Checkpoint dir:", CHECKPOINT_DIR)
print("Using checkpoint:", BEST_CKPT)
print("Available checkpoints:", sorted([p.name for p in CHECKPOINT_DIR.glob("*.pth")]))

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 36 (delta 3), reused 0 (delta 0), pack-reused 23 (from 1)
Receiving objects: 100% (36/36), 47.09 KiB | 3.62 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from src.utils.seed import set_seed
from src.utils.transforms import eval_transform
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, average_macro_f1, CLASS_NAMES
from src.utils.efficiency import measure_fps
from src.utils.wandb_logger import WandbLogger
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.xai.gradcam import GradCAM
from src.models.resnet import resnet18, resnet50
from src.models.vgg import VGG16
from src.models.vit import vit_small_patch16_224

set_seed(42, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
logger = WandbLogger(project=WANDB_PROJECT, run_name="level4-analysis", tags=["level4", "analysis"])

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmg125 (cmg_125) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# data/set_a 가 없으면 zip 을 받아 현재 repo root에 압축 해제 → data/set_a, data/set_b 생성.
import os, sys, zipfile, subprocess
from pathlib import Path

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "aue8088_pa2_data.zip"
EXTRACT_TO = "."   # zip 내부 최상위가 data/ 이므로 repo root에 풀면 data/... 가 됨
DATA_ROOT = "data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# 분석 대상으로 사용할 best 모델 로드 (Level 1 기준 ResNet-18)
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
BEST_CKPT = CHECKPOINT_DIR / "level1_resnet18.pth"

assert CHECKPOINT_DIR.exists(), f"checkpoint directory not found: {CHECKPOINT_DIR}"
assert BEST_CKPT.exists(), f"checkpoint not found: {BEST_CKPT}"

print("Using checkpoint:", BEST_CKPT)
print("Available checkpoints:", sorted([p.name for p in CHECKPOINT_DIR.glob("*.pth")]))

val_ds = BDDAttrDataset(DATA_ROOT, "val", transform=eval_transform())
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

model = resnet18().to(device)
ckpt = torch.load(BEST_CKPT, map_location=device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"])
else:
    model.load_state_dict(ckpt)

model.eval()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터셋이 이미 존재합니다 → ../data/set_a


FileNotFoundError: [Errno 2] No such file or directory: '../checkpoints/level1_resnet18.pth'

In [ ]:
# 속성별 정규화 Confusion Matrix 생성 및 시각화
preds, probs, targets, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, targets, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", ax=ax, cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a])
    ax.set_title(f"{a}")
    ax.set_xlabel("예측 (pred)"); ax.set_ylabel("정답 (true)")
fig.tight_layout()
logger.log_image("analysis/confusion_matrices", fig)

# 속성별 개별 confusion matrix 도 분리해서 업로드
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"analysis/cm_{a}", cms[a], CLASS_NAMES[a])

In [ ]:
# 속성별 Grad-CAM. ResNet 의 경우 마지막 conv 단계를 target_layer 로 사용.
target_layer = model.layer4[-1]
gc = GradCAM(model, target_layer)

batch = next(iter(val_loader))
x = batch["image"][:6].to(device)  # 샘플 이미지 6장

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for row, attr in enumerate(ATTRIBUTES):
    # 각 속성 head 의 최대 logit 합을 score 로 사용 → 해당 head 가 "보는" 영역 추출
    cam = gc(x, lambda out, a=attr: out[a].max(dim=-1).values.sum())
    for col in range(6):
        img = x[col].cpu().permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        axes[row, col].imshow(img)
        axes[row, col].imshow(cam[col].cpu().numpy(), cmap="jet", alpha=0.45)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(attr, fontsize=14)
fig.tight_layout()
logger.log_image("analysis/gradcam_panel", fig)

In [ ]:
# FPS 측정 — 배치 1, 224x224, warm-up 20회 후 200회 평균. 모든 백본에 대해 실행.
fps_rows = []

fps_models = [
    (VGG16, "vgg16", 0.562),
    (resnet18, "resnet18", 0.644),
    (resnet50, "resnet50", 0.625),
    (vit_small_patch16_224, "vit_s16", 0.527),
]

for fn, name, avg_mf1 in fps_models:
    m = fn().to(device).eval()
    fps = measure_fps(m, device, batch_size=1, n_warmup=20, n_iter=200)
    print(f"{name:10s} FPS = {fps:.2f}, Avg-MF1 = {avg_mf1:.3f}")
    fps_rows.append([name, round(fps, 2), avg_mf1])

logger.log_table("analysis/fps", ["backbone", "FPS", "Avg-MF1"], fps_rows)

# 모든 백본에 대해 FPS(x축) vs Avg-MF1(y축) Pareto plot 작성
# 작성한 figure는 logger.log_image("analysis/pareto", fig)로 wandb에 업로드
fig, ax = plt.subplots(figsize=(7, 5))

for name, fps, avg_mf1 in fps_rows:
    ax.scatter(fps, avg_mf1, s=100)
    ax.text(fps, avg_mf1 + 0.005, name, ha="center", fontsize=10)

ax.set_xlabel("FPS (T4 GPU, batch=1)")
ax.set_ylabel("Avg-Macro-F1")
ax.set_title("FPS vs Avg-Macro-F1 Trade-off")
ax.grid(True, alpha=0.3)
fig.tight_layout()

logger.log_image("analysis/pareto", fig)
plt.show()

logger.finish()

In [ ]:
logger.finish()

# Level 5 - Data Mining

Source notebook: `level5_data_mining.ipynb`


# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [1]:
import os
import sys
from pathlib import Path

# 제출 폴더 기준으로 repo root 설정
REPO_ROOT = Path.cwd().resolve()

# 노트북을 notebooks/ 안에서 실행한 경우 한 단계 위를 root로 사용
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

%load_ext autoreload
%autoreload 2

# 의존성 설치
assert os.path.exists("requirements.txt"), "requirements.txt not found. REPO_ROOT 설정을 확인하세요."
!pip install -q -r requirements.txt

# checkpoint 확인
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
assert CHECKPOINT_DIR.exists(), f"checkpoint directory not found: {CHECKPOINT_DIR}"

print("Available checkpoints:", sorted([p.name for p in CHECKPOINT_DIR.glob("*.pth")]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main
Repo root: /content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main
Available checkpoints: ['level1_resnet18.pth', 'level1_resnet18_w211.pth', 'level1_resnet50.pth', 'level1_vgg16.pth', 'level2_vit_s16.pth']


In [2]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "top1k-uncertainty"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cmg125 (cmg_125) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# data/set_a 가 없으면 zip 을 받아 현재 repo root에 압축 해제 → data/set_a, data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "aue8088_pa2_data.zip"
EXTRACT_TO = "."   # zip 내부 최상위가 data/ 이므로 repo root에 풀면 data/... 가 됨
DATA_ROOT = "data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

데이터셋이 이미 존재합니다 → ../data/set_a


In [6]:
# 1단계 — best base 모델로 Set B 의 모든 이미지를 score.
# 현재 저장된 checkpoint 중 가장 안정적인 ResNet-18을 scoring model로 사용한다.

from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
BASE_CKPT = CHECKPOINT_DIR / "level1_resnet18.pth"

assert BASE_CKPT.exists(), f"checkpoint not found: {BASE_CKPT}"

model = resnet18().to(device)
ckpt = torch.load(BASE_CKPT, map_location=device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"])
else:
    model.load_state_dict(ckpt)

model.eval()

set_b = BDDAttrDataset("data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, targets_b, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균.
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)

print("Using checkpoint:", BASE_CKPT)
print(f"Set B size: {len(ids_b)}")
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

Using checkpoint: /content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main/checkpoints/level1_resnet18.pth
Set B size: 15000
unc shape: (15000,), mean=0.146


In [7]:
# 2단계 — Set B 선별 알고리즘
#
# 전략:
#   1) Level 1~4 결과에서 가장 어려웠던 Weather task를 우선 보정
#   2) snowy / foggy / rainy / partly cloudy 등 소수·혼동 클래스에 높은 점수 부여
#   3) dawn/dusk, residential처럼 혼동이 컸던 속성 조합과 모델 uncertainty를 함께 반영
#   4) 한 클래스에 과도하게 몰리지 않도록 weather quota를 적용하여 최대 1,000장 선택

import json
from collections import Counter

K = 1000

weather_names = CLASS_NAMES["weather"]
scene_names = CLASS_NAMES["scene"]
timeofday_names = CLASS_NAMES["timeofday"]

weather_priority = {
    "clear": 0.0,
    "overcast": 0.8,
    "rainy": 1.5,
    "snowy": 2.5,
    "foggy": 2.5,
    "partly cloudy": 1.5,
}

scene_priority = {
    "city street": 0.3,
    "highway": 0.2,
    "residential": 1.0,
}

timeofday_priority = {
    "daytime": 0.0,
    "night": 0.6,
    "dawn/dusk": 1.5,
}

candidates = []

for i, s in enumerate(set_b.samples):
    w = weather_names[int(s.weather)]
    sc = scene_names[int(s.scene)]
    t = timeofday_names[int(s.timeofday)]

    # 기본 점수: 모델 uncertainty + 소수 클래스/어려운 속성 가중치
    score = float(uncertainty[i])
    score += weather_priority[w]
    score += scene_priority[sc]
    score += timeofday_priority[t]

    # 어려운 조합 추가 가산
    if w in ["rainy", "snowy", "foggy"] and t in ["night", "dawn/dusk"]:
        score += 1.0

    if sc == "residential" and t == "dawn/dusk":
        score += 0.7

    candidates.append({
        "idx": i,
        "image_id": s.image_id,
        "weather": int(s.weather),
        "scene": int(s.scene),
        "timeofday": int(s.timeofday),
        "weather_name": w,
        "scene_name": sc,
        "timeofday_name": t,
        "uncertainty": float(uncertainty[i]),
        "score": float(score),
        "reason": "weather minority + difficult combination + uncertainty",
    })

# Weather 기준 quota: minority/어려운 weather를 우선 확보하되 전체 분포가 한쪽으로만 치우치지 않게 제한
weather_quota = {
    "snowy": 220,
    "foggy": 120,
    "rainy": 180,
    "partly cloudy": 180,
    "overcast": 160,
    "clear": 140,
}

picks = []
used_ids = set()

for w_name, quota in weather_quota.items():
    pool = [c for c in candidates if c["weather_name"] == w_name]
    pool = sorted(pool, key=lambda x: x["score"], reverse=True)

    count = 0
    for c in pool:
        if len(picks) >= K:
            break
        if count >= quota:
            break
        if c["image_id"] in used_ids:
            continue

        picks.append(c)
        used_ids.add(c["image_id"])
        count += 1

# quota로 1,000장을 못 채우면 전체 score 순으로 보충
if len(picks) < K:
    for c in sorted(candidates, key=lambda x: x["score"], reverse=True):
        if len(picks) >= K:
            break
        if c["image_id"] in used_ids:
            continue

        picks.append(c)
        used_ids.add(c["image_id"])

# 제출 artifact에는 image_id와 라벨, 선택 근거를 저장
artifact_picks = []
for c in picks:
    artifact_picks.append({
        "image_id": c["image_id"],
        "weather": c["weather"],
        "scene": c["scene"],
        "timeofday": c["timeofday"],
        "weather_name": c["weather_name"],
        "scene_name": c["scene_name"],
        "timeofday_name": c["timeofday_name"],
        "uncertainty": c["uncertainty"],
        "score": c["score"],
        "reason": c["reason"],
    })

print("selected:", len(artifact_picks))
print("Weather distribution:", Counter([p["weather_name"] for p in artifact_picks]))
print("Scene distribution:", Counter([p["scene_name"] for p in artifact_picks]))
print("Time distribution:", Counter([p["timeofday_name"] for p in artifact_picks]))

with open("level5_picks.json", "w", encoding="utf-8") as f:
    json.dump({
        "strategy": (
            "Weather-balanced difficult-combination mining. "
            "Samples are selected by combining model uncertainty, weather minority-class priority, "
            "and difficult scene/time combinations observed in previous levels."
        ),
        "num_picks": len(artifact_picks),
        "picks": artifact_picks,
    }, f, ensure_ascii=False, indent=2)

print(f"saved {len(artifact_picks)} picks to level5_picks.json")

selected: 1000
Weather distribution: Counter({'snowy': 340, 'rainy': 180, 'partly cloudy': 180, 'overcast': 160, 'clear': 140})
Scene distribution: Counter({'city street': 574, 'residential': 298, 'highway': 128})
Time distribution: Counter({'dawn/dusk': 697, 'night': 255, 'daytime': 48})
saved 1000 picks to ../level5_picks.json


In [8]:
# 3단계 — Set A + 본인이 고른 picks 로 fine-tuning.
# 처음부터 학습하지 않고, Level 1 best base checkpoint에서 이어서 학습한다.

import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from collections import Counter

STRATEGY_NAME = "weather_balanced_difficult_uncertainty"
EPOCHS = 15
BATCH = 64

# picks는 2단계에서 선택된 1,000장
extra = [
    (p["image_id"], p["weather"], p["scene"], p["timeofday"])
    for p in picks
]

train_aug = BDDAttrDataset("data/set_a", "train", transform=train_transform(), extra_picks=extra)
val_ds    = BDDAttrDataset("data/set_a", "val",   transform=eval_transform())

g = torch.Generator()
g.manual_seed(SEED)

loader_tr = DataLoader(
    train_aug,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

loader_val = DataLoader(
    val_ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

set_seed(SEED, deterministic=True)

# base model에서 fine-tuning 시작
model2 = resnet18().to(device)

BASE_CKPT = REPO_ROOT / "checkpoints" / "level1_resnet18.pth"
ckpt = torch.load(BASE_CKPT, map_location=device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    model2.load_state_dict(ckpt["state_dict"])
else:
    model2.load_state_dict(ckpt)

optim = torch.optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "resnet18",
        "init_checkpoint": str(BASE_CKPT),
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
        "epochs": EPOCHS,
        "batch": BATCH,
        "lr": 1e-4,
        "weight_decay": 5e-4,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["level5", STRATEGY_NAME],
)

trainer = MultiTaskTrainer(
    model2,
    optim,
    sched,
    losses,
    device,
    TrainConfig(epochs=EPOCHS),
    logger=logger,
)

history = trainer.fit(loader_tr, loader_val)

# 학습 종료 후 — 속성별 confusion matrix 업로드
val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])

# 선택한 1,000장의 실제 라벨 분포 업로드
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

# checkpoint 저장
os.makedirs(REPO_ROOT / "checkpoints", exist_ok=True)

torch.save(
    {
        "state_dict": model2.state_dict(),
        "history": history,
        "strategy": STRATEGY_NAME,
        "num_picks": len(picks),
    },
    REPO_ROOT / "checkpoints" / "level5_final.pth",
)

logger.finish()

print("saved:", REPO_ROOT / "checkpoints" / "level5_final.pth")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/15] train_loss=1.5258  val_avg_MF1=0.6296  per={'weather': 0.5090770596248478, 'scene': 0.6198496311789364, 'timeofday': 0.7598778390559212}


[epoch 02/15] train_loss=1.4696  val_avg_MF1=0.6433  per={'weather': 0.509408058364868, 'scene': 0.6329654425067269, 'timeofday': 0.7874003795066414}


[epoch 03/15] train_loss=1.4329  val_avg_MF1=0.6147  per={'weather': 0.51125602946702, 'scene': 0.5164935108755334, 'timeofday': 0.8162381175003895}


[epoch 04/15] train_loss=1.4030  val_avg_MF1=0.6498  per={'weather': 0.5565897010617068, 'scene': 0.5660911982241622, 'timeofday': 0.8268002606712285}


...

[epoch 13/15] train_loss=1.0982  val_avg_MF1=0.6710  per={'weather': 0.5565299516908213, 'scene': 0.6607263346338901, 'timeofday': 0.7956122372653153}


[epoch 14/15] train_loss=1.0800  val_avg_MF1=0.6777  per={'weather': 0.556691120132726, 'scene': 0.6718734474436899, 'timeofday': 0.8045573128343282}


[epoch 15/15] train_loss=1.0639  val_avg_MF1=0.6698  per={'weather': 0.5390620532477611, 'scene': 0.6658532292495026, 'timeofday': 0.8045573128343282}


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
train/loss,█▇▇▆▆▅▄▄▃▃▂▂▂▁▁
val/avg_macro_f1,▃▄▁▅▃▁▇▄▄▇▇▇▇█▇
val/mf1_scene,▆▆▁▃▃▄▇▅▅▆▇▇▇██
val/mf1_timeofday,▁▄▇█▆▁▅▄▃█▇▇▅▆▆
val/mf1_weather,▃▃▃█▃▁▇▃▄▆▄▆██▆
epoch,15
lr,0
train/loss,1.06393
val/avg_macro_f1,0.66982


saved: /content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main/checkpoints/level5_final.pth


In [9]:
# 4단계 — Kaggle 제출용 CSV 생성.
test_ds = BDDAttrDataset("data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)

os.makedirs("submission", exist_ok=True)
write_submission("submission/level5_submission.csv", ids_te, preds_te)

print("submission/level5_submission.csv 생성 완료")

submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.


In [20]:
import json
import random
from collections import Counter
from pathlib import Path

EPOCHS_ABLATION = 15
BASE_CKPT = REPO_ROOT / "checkpoints" / "level1_resnet18.pth"

assert BASE_CKPT.exists(), f"checkpoint not found: {BASE_CKPT}"


def load_resnet18_from_base():
    model = resnet18().to(device)
    ckpt = torch.load(BASE_CKPT, map_location=device)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    return model


def make_extra_from_picks(pick_list):
    return [
        (p["image_id"], int(p["weather"]), int(p["scene"]), int(p["timeofday"]))
        for p in pick_list
    ]


def make_random_picks(k=1000, seed=SEED):
    rng = random.Random(seed)
    indices = rng.sample(range(len(set_b.samples)), k)

    random_picks = []
    for i in indices:
        s = set_b.samples[i]
        random_picks.append({
            "image_id": s.image_id,
            "weather": int(s.weather),
            "scene": int(s.scene),
            "timeofday": int(s.timeofday),
            "reason": "random baseline",
        })
    return random_picks


def run_level5_compare(run_name, pick_list, epochs=EPOCHS_ABLATION):
    extra = make_extra_from_picks(pick_list)

    train_aug = BDDAttrDataset("data/set_a", "train", transform=train_transform(), extra_picks=extra)
    val_ds = BDDAttrDataset("data/set_a", "val", transform=eval_transform())
    
    g = torch.Generator()
    g.manual_seed(SEED)

    loader_tr = DataLoader(
        train_aug,
        batch_size=64,
        shuffle=True,
        num_workers=2,
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
    )
    loader_val = DataLoader(
        val_ds,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    set_seed(SEED, deterministic=True)
    model = load_resnet18_from_base()

    optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level5-{run_name}",
        config={
            "backbone": "resnet18",
            "strategy": run_name,
            "num_picks": len(pick_list),
            "epochs": epochs,
            "batch": 64,
            "lr": 1e-4,
            "weight_decay": 5e-4,
            "seed": SEED,
            "init_checkpoint": str(BASE_CKPT),
        },
        tags=["level5", run_name],
    )

    trainer = MultiTaskTrainer(
        model,
        optim,
        sched,
        losses,
        device,
        TrainConfig(epochs=epochs),
        logger=logger,
    )

    history = trainer.fit(loader_tr, loader_val)
    metrics = trainer.evaluate(loader_val)

    val_pred, _, val_tgt, _ = collect_predictions(model, loader_val, device)
    cms = confusion_matrices(val_pred, val_tgt)

    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])

        cnt = Counter(p[a] for p in pick_list)
        rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(len(CLASS_NAMES[a]))]
        logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

    ckpt_path = REPO_ROOT / "checkpoints" / f"level5_{run_name}.pth"
    torch.save({"state_dict": model.state_dict(), "history": history, "metrics": metrics}, ckpt_path)

    logger.finish()

    result = {
        "run_name": run_name,
        "num_picks": len(pick_list),
        "avg_macro_f1": float(metrics["avg_macro_f1"]),
        "weather": float(metrics["per_macro_f1"]["weather"]),
        "scene": float(metrics["per_macro_f1"]["scene"]),
        "timeofday": float(metrics["per_macro_f1"]["timeofday"]),
        "checkpoint": str(ckpt_path),
    }

    print(result)
    return result

In [21]:
random_1000 = make_random_picks(k=1000, seed=SEED)

ours_250 = picks[:250]
ours_500 = picks[:500]

results = []

results.append(run_level5_compare("random_1000", random_1000))
results.append(run_level5_compare("ours_250", ours_250))
results.append(run_level5_compare("ours_500", ours_500))

# 이미 돌린 ours_1000 결과를 함께 기록
ours_1000_result = {
    "run_name": "ours_1000",
    "num_picks": 1000,
    "avg_macro_f1": 0.6698241984438639,
    "weather": 0.5390620532477611,
    "scene": 0.6658532292495026,
    "timeofday": 0.8045573128343282,
    "checkpoint": str(REPO_ROOT / "checkpoints" / "level5_final.pth"),
}
results.append(ours_1000_result)

random_avg = results[0]["avg_macro_f1"]
ours_avg = ours_1000_result["avg_macro_f1"]
di_score = (ours_avg - random_avg) / random_avg

summary = {
    "di_formula": "(Avg-MF1_ours - Avg-MF1_random) / Avg-MF1_random",
    "di_score": float(di_score),
    "results": results,
}

with open(REPO_ROOT / "level5_comparison_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("DI score:", di_score)
print(json.dumps(summary, indent=2))

[epoch 01/15] train_loss=1.2825  val_avg_MF1=0.6097  per={'weather': 0.5153315509210511, 'scene': 0.5314770156706518, 'timeofday': 0.7823373381182712}


[epoch 02/15] train_loss=1.2576  val_avg_MF1=0.6072  per={'weather': 0.5067445759621285, 'scene': 0.5209971650437569, 'timeofday': 0.793996245389737}


[epoch 03/15] train_loss=1.2337  val_avg_MF1=0.6662  per={'weather': 0.5627309463879043, 'scene': 0.6328376970040379, 'timeofday': 0.8029373522458628}


[epoch 04/15] train_loss=1.1920  val_avg_MF1=0.6031  per={'weather': 0.5178346230981102, 'scene': 0.5135696843478118, 'timeofday': 0.7780418317003682}


[epoch 05/15] train_loss=1.1794  val_avg_MF1=0.6502  per={'weather': 0.5317687134414585, 'scene': 0.5819577739158762, 'timeofday': 0.836996336996337}


...

[epoch 13/15] train_loss=0.9967  val_avg_MF1=0.6713  per={'weather': 0.5567273084559792, 'scene': 0.6644081974895221, 'timeofday': 0.7927243666001696}


[epoch 14/15] train_loss=0.9643  val_avg_MF1=0.6771  per={'weather': 0.5565787584456187, 'scene': 0.6718124077769941, 'timeofday': 0.8029441729655863}


[epoch 15/15] train_loss=0.9542  val_avg_MF1=0.6750  per={'weather': 0.544694615904394, 'scene': 0.6739895066933502, 'timeofday': 0.8062678062678063}


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,██▇▇▆▆▅▄▃▃▂▂▁▁▁
train/loss,█▇▇▆▆▅▅▄▄▃▂▂▂▁▁
val/avg_macro_f1,▄▄▄▆▄▁▄▅▄▆▇████
val/mf1_scene,▁▄▇▇▅▂▃▃▃▆▇█▇██
val/mf1_timeofday,▆▁▂▅▃▃█▇▄▄▃▅▄▅▅
val/mf1_weather,█▆▂▆▅▁▄▇▅▇▇▇██▇
epoch,15
lr,0
train/loss,0.95423
val/avg_macro_f1,0.67498


{'run_name': 'ours_500', 'num_picks': 500, 'avg_macro_f1': 0.6749839762885168, 'weather': 0.544694615904394, 'scene': 0.6739895066933502, 'timeofday': 0.8062678062678063, 'checkpoint': '/content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main/checkpoints/level5_ours_500.pth'}
DI score: 0.013600644973442414
{
  "di_formula": "(Avg-MF1_ours - Avg-MF1_random) / Avg-MF1_random",
  "di_score": 0.013600644973442414,
  "results": [
    {
      "run_name": "random_1000",
      "num_picks": 1000,
      "avg_macro_f1": 0.6608363972197493,
      "weather": 0.5470300350232594,
      "scene": 0.6356948775413643,
      "timeofday": 0.799784279094624,
      "checkpoint": "/content/2026-HYU-AUE8088-PA2-main/2026-HYU-AUE8088-PA2-main/checkpoints/level5_random_1000.pth"
    },
    {
      "run_name": "ours_250",
      "num_picks": 250,
      "avg_macro_f1": 0.6489975681149959,
      "weather": 0.5253587543715502,
      "scene": 0.6288485586160814,
      "timeofday": 0.7927853913573558,
      "checkp

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.